In [47]:
using LowLevelFEM
using LinearAlgebra
using SparseArrays
using Test

spectral_path = joinpath(
    dirname(pathof(LowLevelFEM)),
    "eigen.jl"
)

@assert isfile(spectral_path) "Copy spectral.jl into the LowLevelFEM src directory first: $spectral_path"

Base.include(
    LowLevelFEM,
    spectral_path
)

using LowLevelFEM: spectralTransformation

┌ Warning: Replacing docs for `LowLevelFEM.prepare_eigen_system :: Tuple{Union{LowLevelFEM.SymmetricSystemMatrix, SystemMatrix}, Union{LowLevelFEM.SymmetricSystemMatrix, SystemMatrix}}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._eigen_selected_indices :: Tuple{Any, Int64}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM._eigen_nearly_symmetric :: Tuple{Any}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM.reconstruct_eigenvectors :: Tuple{Any, Any}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM.solveEigen :: Tuple{Union{LowLevelFEM.SymmetricSystemMatrix, SystemMatrix}, Union{LowLevelFEM.SymmetricSystemMatrix, SystemMatrix}}` in module `LowLevelFEM`
└ @ Base.Docs docs/Docs.jl:253
┌ Warning: Replacing docs for `LowLevelFEM.solveEigenProblem :: Tuple{Union{LowLevelFEM.SymmetricSystemM

In [48]:
structured_rect_mesh(lx=10)

mat = Material("body")
U = Field([mat], type=:VectorField, dim=2, fieldName=:u);

In [49]:
ndofs(U)

2222

In [50]:
K = ∫(ε(U) ⋅ D(:PlaneStress, mat) ⋅ ε(U))
M = ∫(U ⋅ mat.ρ ⋅ U);

In [51]:
bc = BoundaryCondition("left", ux=0, uy=0)

BoundaryCondition("left", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :ux => 0))

In [52]:
λ, Φ = solveEigen(Symmetric(K), Symmetric(M), solver=:direct)
λ

6-element Vector{Float64}:
 -2.6879879807254734
 -2.327613366938404
  1.255336890529273
  9.976753958142934e10
  6.775378057879274e11
  2.264829897568584e12

In [53]:
λ, Φ = solveEigen(Symmetric(K), Symmetric(M), solver=:arpack)
λ

6-element Vector{Float64}:
 -0.28050489136658513
 -0.007145488788418331
 11.864596072455527
  9.976753958145676e10
  6.775378057865458e11
  2.2648298975705923e12

In [54]:
λ, Φ = solveEigen(Symmetric(K), Symmetric(M), support=[bc], solver=:direct)
λ

6-element Vector{Float64}:
 2.603546177375141e9
 9.376097657421771e10
 6.302018177369926e11
 6.512032800501093e11
 2.1534328737267305e12
 5.016371047742017e12

In [55]:
λ, Φ = solveEigen(Symmetric(K), Symmetric(M), shift=0, nev=4, solver=:arpack)
λ

4-element Vector{Float64}:
 9.97675357771204e10
 6.775378742280765e11
 2.2648298800048403e12
 2.512904689950051e12

In [56]:
λ, Φ = solveEigen(Symmetric(K), Symmetric(M), support=[bc])
λ

6-element Vector{Float64}:
 2.6035461792632976e9
 9.37609765770908e10
 6.302018177356112e11
 6.512032800496271e11
 2.1534328737256978e12
 5.016371047743216e12

In [57]:
λ, ϕ = solveEigen(
    Symmetric(K),
    Symmetric(M);
    support=[bc],
    n=6,
    shift=0.0,
    solver=:arpack,
    solveroptions=(ncv=40, maxiter=10000)
)

λ

6-element Vector{Float64}:
 2.603546179428856e9
 9.376097657723308e10
 6.30201817738631e11
 6.512032800499176e11
 2.1534328737271072e12
 5.016371047742172e12

In [58]:
λ, ϕ = solveEigen(
    Symmetric(K),
    Symmetric(M);
    support=[bc],
    n=6,
    sigma=0.0,
    which=:LM,
    solver=:arpack,
    solveroptions=(
        explicittransform=:none,
        ncv=40,
        maxiter=10000
    )
)

@show λ
@show sqrt.(real.(λ)) ./ (2π)

λ = [2.6035461792842393e9, 9.376097657714363e10, 6.30201817738538e11, 6.512032800498846e11, 2.1534328737270862e12, 5.016371047742034e12]
sqrt.(real.(λ)) ./ (2π) = [8120.874037820148, 48733.90279430147, 126345.5520653789, 128433.53059698534, 233553.188621109, 356463.410515854]


6-element Vector{Float64}:
   8120.874037820148
  48733.90279430147
 126345.5520653789
 128433.53059698534
 233553.188621109
 356463.410515854

In [59]:
λ, ϕ = solveEigen(Symmetric(K), Symmetric(M), sigma=1, which=:LM)
λ

6-element Vector{Float64}:
 -0.1926102980921145
 -0.15806082920636522
 -0.09786844977501685
  9.97675395829497e10
  6.775378057853064e11
  2.264829897570235e12

In [60]:
λ, ϕ = solveEigen(K, M, support=[bc])
λ

6-element Vector{Float64}:
 2.603546177058712e9
 9.376097657683444e10
 6.302018177387987e11
 6.512032800504329e11
 2.1534328737266506e12
 5.016371047741624e12

In [61]:
P = LowLevelFEM.prepare_eigen_system(
    K, M;
    support=[bc]
)

Kr, Mr = P.A, P.B

@show size(Kr)
@show nnz(Kr), nnz(Mr)

@show norm(Kr - Kr')
@show norm(Mr - Mr')

@show extrema(diag(Kr))
@show extrema(diag(Mr))

@show isposdef(Symmetric(Kr))
@show isposdef(Symmetric(Mr))

size(Kr) = (2200, 2200)
(nnz(Kr), nnz(Mr)) = (36815, 18476)
norm(Kr - Kr') = 7.624083117896295e-10
norm(Mr - Mr') = 0.0
extrema(diag(Kr)) = (98901.09890109849, 395604.39560440095)
extrema(diag(Mr)) = (8.722222222222265e-12, 3.488888888888941e-11)
isposdef(Symmetric(Kr)) = true
isposdef(Symmetric(Mr)) = true


true

In [62]:
per = MultiPointConstraint(master="left", slave="right")

MultiPointConstraint("left", "right", nothing, Dict{Symbol, Bool}())

In [63]:
e1 = VectorField(U, "left", [-1, 0, 0])
cs = CoordinateSystem(e1)

CoordinateSystem(VectorField([[-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;]], Matrix{Float64}(undef, 0, 0), [0.0], [215, 216, 217, 218, 219, 220, 221, 222, 223, 224], 1, :v3D, Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1111, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing))), nothing, 1.0)

In [67]:
λ, Φ = solveEigen(Symmetric(K), Symmetric(M), coordSys=[cs], mpc=[per], solver=:direct)
λ

6-element Vector{Float64}:
 2.0129937711412113e10
 2.012993771605723e10
 1.3155147134091475e12
 1.3155147134114658e12
 2.5129049216648975e12
 2.5129049216735977e12

In [68]:
λ, Φ = solveEigen(Symmetric(K), Symmetric(M), coordSys=[cs], mpc=[per])
λ

6-element Vector{Float64}:
 2.0129937714612267e10
 2.0129937720079075e10
 1.3155147134111023e12
 1.3155147134121104e12
 2.5129049216654595e12
 2.5129049216697163e12

In [65]:
λ, Φ = solveEigen(Symmetric(K), Symmetric(M), coordSys=[cs], mpc=[per], solveroptions=(ncv=40, maxiter=10000))
λ

6-element Vector{Float64}:
 2.0129937717951942e10
 2.012993771817165e10
 1.3155147134128262e12
 1.3155147134135442e12
 2.512904921662524e12
 2.5129049216666426e12